# Document Chunking Masterclass — From Fixed-Size to Agentic

**Goal:** understand every major chunking strategy used in production RAG systems,
by running each one on the *same* real documents and comparing what comes out.

**Inputs used throughout this notebook:**
- `data/clinical_trial_protocol.pdf` — a 5-page clinical trial protocol (headings, lists, numbered sections)
- `data/data_handling_policy.docx` — a compliance policy document (headings, bullet lists)

**Roadmap:**
1. Setup — load and inspect the raw text of both documents
2. Fixed-size chunking
3. Sentence / Paragraph chunking
4. Recursive character chunking
5. Structure-aware chunking
6. Semantic chunking
7. LLM / Agentic chunking
8. Build a vector store per strategy + run 5 fixed questions against each
9. Final comparison table + healthcare-specific recommendations

We build this one section at a time — run each cell, look at the actual output,
*then* move to the next strategy. Don't skip ahead; the whole point is to build
intuition for *why* chunk boundaries end up where they do.


## Section 0 — Setup: Loading Our Two Real Documents

Before we can chunk anything, we need plain text. Real documents are messy:
- PDFs store text as positioned glyphs, not paragraphs — extraction can merge or split lines oddly.
- DOCX files store text in "runs" inside XML — headings and bullets carry structure metadata we can use later (Section 5).

**Production note:** in a real pipeline this "extraction" step is its own component,
often the least glamorous and most bug-prone part of RAG. Garbage extraction → garbage chunks →
garbage retrieval, no matter how good your chunking strategy is downstream.


In [1]:
# If running this notebook fresh, install dependencies first (uncomment):
# %pip install pypdf python-docx langchain langchain-text-splitters tiktoken \
#              sentence-transformers faiss-cpu scikit-learn nltk

import os
import textwrap


DATA_DIR = "../../../data"
PDF_PATH = os.path.join(DATA_DIR, "clinical_trial_protocol.pdf")
DOCX_PATH = os.path.join(DATA_DIR, "data_handling_policy.docx")

print("PDF exists:", os.path.exists(PDF_PATH))
print("DOCX exists:", os.path.exists(DOCX_PATH))


PDF exists: True
DOCX exists: True


### 0.1 Extract text from the PDF

In [2]:
from pypdf import PdfReader

def load_pdf_text(path: str) -> str:
    """Extract raw text from a PDF, page by page, keeping a page marker.
    We keep '[[PAGE:n]]' markers so structure-aware chunking (Section 5)
    can later use page boundaries if it needs to.
    """
    reader = PdfReader(path)
    pages_text = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        pages_text.append(f"[[PAGE:{i+1}]]\n{text}")
    return "\n".join(pages_text)

pdf_text = load_pdf_text(PDF_PATH)

print(f"Total characters extracted: {len(pdf_text):,}")
print(f"Approx. words: {len(pdf_text.split()):,}")
print("\n--- First 600 characters ---\n")
print(pdf_text[:600])


Total characters extracted: 10,653
Approx. words: 1,536

--- First 600 characters ---

[[PAGE:1]]
Clinical Trial Protocol
Protocol Number: CTP-2026-0142
Study Title: A Phase II, Randomized, Double-Blind, Placebo-Controlled Study of Oral Compound AX-119
in Adult Patients with Moderate-to-Severe Rheumatoid Arthritis
Sponsor: Meridian Biotherapeutics Inc.
Version: 3.1 | Date: 15-Jan-2026
1. Study Objectives
1.1 Primary Objective
The primary objective of this study is to evaluate the efficacy of AX-119 compared to placebo in reducing
disease activity, as measured by the American College of Rheumatology 20% response criteria (ACR20),
at Week 24 in adult patients with moderate-to-seve


### 0.2 Extract text from the DOCX

In [3]:
from docx import Document as DocxDocument

def load_docx_text(path: str):
    """Extract text paragraph by paragraph, keeping each paragraph's
    style name (e.g. 'Heading 1', 'List Bullet', 'Normal'). We return a
    list of (style, text) tuples — this structure is what Section 5
    (structure-aware chunking) will use directly, and we also build a
    plain joined string for the strategies that just want raw text.
    """
    doc = DocxDocument(path)
    paragraphs = []
    for para in doc.paragraphs:
        text = para.text.strip()
        if not text:
            continue
        # Real-world gotcha: para.style can be None if the style reference
        # doesn't resolve cleanly. Don't let that crash extraction -- fall
        # back to "Normal" and move on. Bad extraction should degrade
        # gracefully, not blow up the whole pipeline.
        style_name = para.style.name if para.style is not None else "Normal"
        paragraphs.append((style_name, text))
    return paragraphs

docx_paragraphs = load_docx_text(DOCX_PATH)
docx_text = "\n".join(text for _, text in docx_paragraphs)

print(f"Total paragraphs: {len(docx_paragraphs)}")
print(f"Total characters: {len(docx_text):,}")
print("\n--- First 10 (style, text) pairs ---\n")
for style, text in docx_paragraphs[:10]:
    print(f"[{style}] {textwrap.shorten(text, width=80)}")


Total paragraphs: 31
Total characters: 4,707

--- First 10 (style, text) pairs ---

[Title] Clinical Data Handling and Privacy Policy
[Normal] Document ID: POL-COMP-014 | Version 2.3 | Effective Date: 01-Feb-2026
[Heading 1] 1. Purpose
[Normal] This policy establishes the requirements for the collection, storage, [...]
[Heading 1] 2. Scope
[Normal] This policy covers protected health information (PHI), personally [...]
[Heading 1] 3. Regulatory Framework
[Normal] Data handling practices under this policy must comply with the following [...]
[List Paragraph] The Health Insurance Portability and Accountability Act (HIPAA) for data [...]
[List Paragraph] The General Data Protection Regulation (GDPR) for data collected within [...]


### 0.3 A quick sanity check

Before chunking, always eyeball the raw extracted text. Two things to check:

1. **Did extraction lose anything obvious?** (tables collapsing, headers merging into body text, garbled characters)
2. **Roughly how big is the document?** — this tells you whether "chunking" even matters (a 2-page doc barely needs it) and what chunk size might be sensible relative to total length.


In [5]:
print("=" * 60)
print("PDF  — clinical_trial_protocol.pdf")
print("=" * 60)
print(f"Characters: {len(pdf_text):,} | Words: {len(pdf_text.split()):,} | Pages: {pdf_text.count('[[PAGE:')}")

print()
print("=" * 60)
print("DOCX — data_handling_policy.docx")
print("=" * 60)
print(f"Characters: {len(docx_text):,} | Words: {len(docx_text.split()):,} | Paragraphs: {len(docx_paragraphs)}")


PDF  — clinical_trial_protocol.pdf
Characters: 10,653 | Words: 1,536 | Pages: 5

DOCX — data_handling_policy.docx
Characters: 4,707 | Words: 681 | Paragraphs: 31


**What we have now:**
- `pdf_text` — full plain text of the clinical trial protocol, with `[[PAGE:n]]` markers
- `docx_text` — full plain text of the compliance policy, paragraphs joined by newline
- `docx_paragraphs` — list of `(style_name, text)` — keeps Word's own structure info intact

Everything from Section 1 onward will chunk these same variables, so every
strategy is compared on identical input — that's what makes the final
comparison meaningful.

---
✅ **Checkpoint — Section 0 complete.**
Confirm this looks right, and we'll move to **Section 1: Fixed-size chunking**.


## Section 1 — Fixed-size Chunking

**The idea in plain words:** take the text, and cut it into pieces of the same length,
counting a fixed number of characters (or tokens) at a time. That's it. It doesn't look
at sentences, paragraphs, or headings — it just counts and cuts.

Most implementations also let neighboring chunks **overlap** a little, so a sentence that
gets cut in half at a chunk boundary still appears in full in at least one chunk.

### How it works (ASCII picture)

Imagine the text as one long strip. `chunk_size = 20` characters, `overlap = 5` characters:

```
Full text:  |----------------------------------------------------|
             0         10        20        30        40        50

Chunk 1:    |--------------------|
             0                   20

Chunk 2:              |--------------------|
                       15                  35   <- starts 5 chars before chunk 1 ended (the overlap)

Chunk 3:                        |--------------------|
                                 30                  50
```

Each new chunk starts `chunk_size - overlap` characters after the previous chunk started.
That's the entire algorithm — no understanding of the content at all.


### 1.1 Build it from scratch (so the mechanism is fully visible)

In [4]:
def fixed_size_chunks(text: str, chunk_size: int = 500, overlap: int = 50):
    """Cut `text` into pieces of `chunk_size` characters, each new piece
    starting `chunk_size - overlap` characters after the previous one started.
    No awareness of words, sentences, or structure -- purely a character count.
    """
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    step = chunk_size - overlap
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += step
    return chunks


# Try it on the PDF text
fixed_chunks_pdf = fixed_size_chunks(pdf_text, chunk_size=500, overlap=50)

print(f"Number of chunks: {len(fixed_chunks_pdf)}")
print(f"Chunk sizes (chars): {[len(c) for c in fixed_chunks_pdf[:8]]} ...")


Number of chunks: 24
Chunk sizes (chars): [500, 500, 500, 500, 500, 500, 500, 500] ...


### 1.2 Look at the actual chunks (this is the part that matters)

In [5]:
for i, chunk in enumerate(fixed_chunks_pdf[:4]):
    print(f"--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)
    print()


--- Chunk 1 (500 chars) ---
[[PAGE:1]]
Clinical Trial Protocol
Protocol Number: CTP-2026-0142
Study Title: A Phase II, Randomized, Double-Blind, Placebo-Controlled Study of Oral Compound AX-119
in Adult Patients with Moderate-to-Severe Rheumatoid Arthritis
Sponsor: Meridian Biotherapeutics Inc.
Version: 3.1 | Date: 15-Jan-2026
1. Study Objectives
1.1 Primary Objective
The primary objective of this study is to evaluate the efficacy of AX-119 compared to placebo in reducing
disease activity, as measured by the American Colle

--- Chunk 2 (500 chars) ---
isease activity, as measured by the American College of Rheumatology 20% response criteria (ACR20),
at Week 24 in adult patients with moderate-to-severe rheumatoid arthritis who have had an inadequate
response to methotrexate.
1.2 Secondary Objectives

To assess the safety and tolerability of AX-119 over 24 weeks of treatment.

To evaluate the proportion of patients achieving ACR50 and ACR70 response at Week 24.

To assess change from 

**Look closely at chunk 2 and 3 above.** Notice how a chunk can start or end
*mid-sentence*, or even mid-word, because the cut is purely a character count --
it has no idea what a sentence is. This is the central weakness of fixed-size
chunking, and it's the whole reason the next five sections exist.

The overlap (50 chars here) means chunk boundaries share some text, so an idea
that got cut in half in chunk 2 is more likely to survive whole in chunk 3.
But overlap is a patch, not a fix -- it reduces how often meaning gets broken,
it doesn't stop it from happening.


### 1.3 Why character count isn't the whole story: tokens

In [6]:
import tiktoken

# Embedding models and LLMs don't read "characters" -- they read "tokens",
# which are roughly (but not exactly) sub-word pieces. A 500-character chunk
# is NOT the same as a 500-token chunk. Let's see the actual difference.
#
# tiktoken downloads its vocabulary file from the internet the first time
# it's used, then caches it locally. If you're on a machine with no internet
# access at all, this falls back to a rough whitespace-based approximation
# so the rest of the notebook still runs -- but the real tiktoken encoding
# is what you want for production accuracy.
try:
    encoding = tiktoken.get_encoding("cl100k_base")  # used by many OpenAI/Claude-era tokenizers as an approximation
    USING_REAL_TOKENIZER = True
except Exception as e:
    print(f"Could not download tiktoken vocabulary ({type(e).__name__}). "
          f"Falling back to a simple whitespace-based approximation.")

    class _FallbackEncoding:
        def encode(self, text):
            return text.split(" ")
        def decode(self, tokens):
            return " ".join(tokens)

    encoding = _FallbackEncoding()
    USING_REAL_TOKENIZER = False

sample = fixed_chunks_pdf[0]
num_chars = len(sample)
num_tokens = len(encoding.encode(sample))

print(f"Sample chunk: {num_chars} characters -> {num_tokens} tokens")
print(f"Rough ratio: {num_chars / num_tokens:.2f} characters per token")


Sample chunk: 500 characters -> 132 tokens
Rough ratio: 3.79 characters per token


**Why this matters in production:** embedding models and LLMs have a maximum
**token** limit per request, not a character limit. If you size your chunks by
character count only, you can accidentally build a chunk that's too big for
the model's context window, especially with text that has lots of numbers,
abbreviations, or non-English words (those tend to use more tokens per
character). Production chunkers almost always measure size in tokens, not characters.

Let's redo the same fixed-size chunking, but counting tokens instead of characters.


In [8]:
def fixed_size_chunks_by_tokens(text: str, chunk_size_tokens: int = 150, overlap_tokens: int = 20,
                                 enc=encoding):
    """Same sliding-window idea as before, but the ruler is 'tokens' instead
    of 'characters'. We encode the whole text to token ids once, slide the
    window over the token ids, then decode each window back to text.
    """
    if overlap_tokens >= chunk_size_tokens:
        raise ValueError("overlap_tokens must be smaller than chunk_size_tokens")

    token_ids = enc.encode(text)
    step = chunk_size_tokens - overlap_tokens
    chunks = []
    start = 0
    while start < len(token_ids):
        end = start + chunk_size_tokens
        window = token_ids[start:end]
        chunks.append(enc.decode(window))
        start += step
    return chunks


fixed_chunks_pdf_tok = fixed_size_chunks_by_tokens(pdf_text, chunk_size_tokens=150, overlap_tokens=20)
print(f"Number of chunks (token-based): {len(fixed_chunks_pdf_tok)}")
print()
print("--- Chunk 1 ---")
print(fixed_chunks_pdf_tok[0])


Number of chunks (token-based): 19

--- Chunk 1 ---
[[PAGE:1]]
Clinical Trial Protocol
Protocol Number: CTP-2026-0142
Study Title: A Phase II, Randomized, Double-Blind, Placebo-Controlled Study of Oral Compound AX-119
in Adult Patients with Moderate-to-Severe Rheumatoid Arthritis
Sponsor: Meridian Biotherapeutics Inc.
Version: 3.1 | Date: 15-Jan-2026
1. Study Objectives
1.1 Primary Objective
The primary objective of this study is to evaluate the efficacy of AX-119 compared to placebo in reducing
disease activity, as measured by the American College of Rheumatology 20% response criteria (ACR20),
at Week 24 in


### 1.4 Same strategy on the DOCX (compliance policy)

In [10]:
fixed_chunks_docx = fixed_size_chunks(docx_text, chunk_size=500, overlap=50)

print(f"Number of chunks: {len(fixed_chunks_docx)}")
print()
for i, chunk in enumerate(fixed_chunks_docx[:3]):
    print(f"--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)
    print()


Number of chunks: 11

--- Chunk 1 (500 chars) ---
Clinical Data Handling and Privacy Policy
Document ID: POL-COMP-014   |   Version 2.3   |   Effective Date: 01-Feb-2026
1. Purpose
This policy establishes the requirements for the collection, storage, processing, and disclosure of clinical trial participant data across all studies sponsored or managed by Meridian Biotherapeutics Inc. It applies to all employees, contractors, clinical research organizations (CROs), and third-party vendors who handle participant data on behalf of the company.
2. S

--- Chunk 2 (500 chars) ---
le participant data on behalf of the company.
2. Scope
This policy covers protected health information (PHI), personally identifiable information (PII), and any derived clinical or genomic data collected during the conduct of clinical trials, observational studies, and post-marketing surveillance activities. It applies regardless of the format in which the data is stored, including electronic health records, case rep

**Same problem shows up here too.** The compliance policy has 10 clearly numbered
sections (Purpose, Scope, Regulatory Framework, ...), each meant to stand alone as
one idea. Fixed-size chunking has no idea those section boundaries exist -- it will
happily glue the end of "Section 3: Regulatory Framework" to the start of
"Section 4: Data Classification" in the same chunk, or split Section 4 into two
separate chunks that each lose context about what section they belong to.

For a compliance document, that's a real problem: if a compliance officer asks
*"what's our breach notification deadline?"*, you want the retrieved chunk to be
the complete Section 8, not half of Section 7 stitched to the first third of Section 8.

### Pros

- **Dead simple to implement and reason about.** One function, two parameters.
- **Fast.** No NLP, no embeddings, no model calls -- just counting.
- **Deterministic and predictable chunk sizes.** Useful when you have a hard token
  budget (e.g. must fit under a strict embedding model limit) and predictability
  matters more than chunk quality.
- **Language-agnostic.** Works the same regardless of what language the text is in
  (as long as you're measuring in tokens, not assuming whitespace-separated words).

### Cons

- **Breaks sentences and ideas mid-way**, as we just saw.
- **Ignores document structure entirely** -- headings, lists, tables mean nothing to it.
- **Overlap wastes storage and compute** (the same text gets embedded more than once)
  without fully solving the boundary problem.
- **Retrieval quality suffers** -- a half-sentence chunk often doesn't contain enough
  meaning to match a user's question well, and doesn't read well if shown to a user.

### When this comes up in interviews

This is usually the "naive baseline" interviewers expect you to know and then
critique. A common question: *"Why wouldn't you just use fixed-size chunking for
a production RAG system handling contracts/medical records/etc.?"* — the answer
they're looking for is exactly what we just demonstrated: it breaks semantic units,
which hurts retrieval precision. Good follow-up to mention: fixed-size chunking
is often still used as a **fallback layer inside** smarter strategies (e.g. "split
by paragraph, but if a paragraph is still too big, fall back to fixed-size splitting
within it") -- which is exactly what Recursive Character chunking does in Section 3.

---
✅ **Checkpoint — Section 1 complete.**
We now have `fixed_chunks_pdf` and `fixed_chunks_docx` saved as our baseline.
Every later strategy will get compared back to what we just saw here.
Confirm this looks right, and we'll move to **Section 2: Sentence / Paragraph chunking**.


## Section 2 — Sentence / Paragraph Chunking

**The idea in plain words:** instead of cutting by a fixed character count, cut at
natural language boundaries -- the end of a sentence, or the end of a paragraph --
and then group those pieces together until we hit roughly our target chunk size.
The cut points are chosen by the *content*, not by a ruler.

### How it works (ASCII picture)

```
Fixed-size:      |------cut------|------cut------|------cut------|
                       (ignores sentence endings entirely)

Sentence-based:  [Sentence 1.] [Sentence 2.] [Sentence 3.] [Sentence 4.] ...
                  \___________chunk 1___________/ \____chunk 2____/
                  (grouped up to ~size limit, but a cut only ever happens
                   AFTER a full sentence, never in the middle of one)
```

We never slice through the middle of a sentence. If adding the next sentence would
push a chunk over the size limit, we close the current chunk and start a new one
with that sentence.

**One honest limitation worth flagging up front:** PDFs don't reliably preserve
paragraph breaks the way Word documents do -- when a PDF is generated, "paragraph"
information (a blank line, an indent) often isn't stored as text at all, just as
visual spacing. Our `pdf_text` doesn't have blank lines between paragraphs. Word
documents, on the other hand, store each paragraph as its own explicit XML element
-- which is exactly what `docx_paragraphs` gave us in Section 0. So:
- We'll demonstrate **sentence chunking** on the PDF (sentences survive extraction fine).
- We'll demonstrate **paragraph chunking** on the DOCX (paragraphs are explicit there).


### 2.1 Sentence chunking (on the PDF)

In [10]:
import re

def split_into_sentences(text: str):
    """A simple, dependency-free sentence splitter: cut after '.', '?', or '!'
    when followed by whitespace and a capital letter (or end of text). Not as
    accurate as a real NLP sentence tokenizer (spaCy, NLTK's punkt), which handle
    abbreviations like 'Dr.' or 'e.g.' properly -- but it needs no model download,
    so it always works offline. Good enough to see the concept clearly.
    """
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    # Split, keeping the sentence-ending punctuation attached
    sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z\[])", text)
    return [s.strip() for s in sentences if s.strip()]


def sentence_chunks(text: str, max_chars: int = 500):
    """Group whole sentences together until adding the next one would exceed
    max_chars, then start a new chunk. Never splits a sentence in half.
    """
    sentences = split_into_sentences(text)
    chunks, current = [], ""
    for sent in sentences:
        candidate = (current + " " + sent).strip() if current else sent
        if len(candidate) > max_chars and current:
            chunks.append(current)
            current = sent
        else:
            current = candidate
    if current:
        chunks.append(current)
    return chunks


sentence_chunks_pdf = sentence_chunks(pdf_text, max_chars=500)

print(f"Number of chunks: {len(sentence_chunks_pdf)}")
print(f"Chunk sizes (chars): {[len(c) for c in sentence_chunks_pdf[:8]]} ...")
print()
for i, chunk in enumerate(sentence_chunks_pdf[:3]):
    print(f"--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)
    print()


Number of chunks: 24
Chunk sizes (chars): [303, 734, 439, 427, 331, 305, 406, 1218] ...

--- Chunk 1 (303 chars) ---
[[PAGE:1]] Clinical Trial Protocol Protocol Number: CTP-2026-0142 Study Title: A Phase II, Randomized, Double-Blind, Placebo-Controlled Study of Oral Compound AX-119 in Adult Patients with Moderate-to-Severe Rheumatoid Arthritis Sponsor: Meridian Biotherapeutics Inc. Version: 3.1 | Date: 15-Jan-2026 1.

--- Chunk 2 (734 chars) ---
Study Objectives 1.1 Primary Objective The primary objective of this study is to evaluate the efficacy of AX-119 compared to placebo in reducing disease activity, as measured by the American College of Rheumatology 20% response criteria (ACR20), at Week 24 in adult patients with moderate-to-severe rheumatoid arthritis who have had an inadequate response to methotrexate. 1.2 Secondary Objectives  To assess the safety and tolerability of AX-119 over 24 weeks of treatment.  To evaluate the proportion of patients achieving ACR50 and ACR70 respons

**Compare this to Section 1.** Chunk 1 in Section 1 ended mid-sentence
(`"...American Colle"`). Here, every chunk ends on a complete sentence -- no
more words getting sliced in half. That's the entire benefit of this strategy,
and it's a real, meaningful improvement for retrieval quality: a chunk that's
a whole thought reads better and embeds more meaningfully than a fragment.


### 2.2 Paragraph chunking (on the DOCX)

In [11]:
def paragraph_chunks(paragraphs, max_chars: int = 500):
    """paragraphs is a list of (style, text) tuples, e.g. from docx_paragraphs.
    Group whole paragraphs together until the size limit, same logic as sentence
    chunking but the unit is a paragraph instead of a sentence. If a single
    paragraph is already bigger than max_chars, it becomes its own chunk
    (we don't split inside a paragraph here -- see Section 3 for what handles that).
    """
    chunks, current = [], ""
    for _, text in paragraphs:
        candidate = (current + "\n" + text).strip() if current else text
        if len(candidate) > max_chars and current:
            chunks.append(current)
            current = text
        else:
            current = candidate
    if current:
        chunks.append(current)
    return chunks


paragraph_chunks_docx = paragraph_chunks(docx_paragraphs, max_chars=500)

print(f"Number of chunks: {len(paragraph_chunks_docx)}")
print()
for i, chunk in enumerate(paragraph_chunks_docx[:4]):
    print(f"--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)
    print()


Number of chunks: 12

--- Chunk 1 (495 chars) ---
Clinical Data Handling and Privacy Policy
Document ID: POL-COMP-014   |   Version 2.3   |   Effective Date: 01-Feb-2026
1. Purpose
This policy establishes the requirements for the collection, storage, processing, and disclosure of clinical trial participant data across all studies sponsored or managed by Meridian Biotherapeutics Inc. It applies to all employees, contractors, clinical research organizations (CROs), and third-party vendors who handle participant data on behalf of the company.

--- Chunk 2 (446 chars) ---
2. Scope
This policy covers protected health information (PHI), personally identifiable information (PII), and any derived clinical or genomic data collected during the conduct of clinical trials, observational studies, and post-marketing surveillance activities. It applies regardless of the format in which the data is stored, including electronic health records, case report forms, biological samples, and imaging data.
3.

**Compare to Section 1's fixed-size chunking on the same DOCX.** There, chunk 2
started mid-word (`"le participant data..."`, sliced out of "the"). Here, every
chunk boundary lines up with a real paragraph boundary -- notice Chunk 1 above
naturally groups the title, the document ID line, and "1. Purpose" with its full
body text together, because none of those paragraphs individually pushed past
500 characters.

### Pros

- **Never breaks a sentence or paragraph in half** -- immediate, visible quality
  improvement over fixed-size, at almost no extra implementation cost.
- **Still simple** -- no embeddings, no model calls, just splitting logic.
- **Chunks read naturally** if shown to a user (e.g. as a citation snippet).

### Cons

- **Chunk sizes become uneven.** One "sentence" chunk might be 80 characters,
  another might be 490 -- harder to reason about a hard token budget.
- **Still purely structural, not meaning-aware.** It doesn't know that two
  consecutive sentences discuss completely different topics, or that two
  paragraphs on the same page are actually one continuous idea split by a
  page break. It only respects grammar boundaries, not topic boundaries.
- **Real sentence splitting is harder than it looks.** Our regex splitter will
  get confused by abbreviations ("Dr.", "e.g.", "Fig. 3", "eGFR of 40 mL/min/1.73m^2.")
  -- a production system usually reaches for spaCy or NLTK's trained sentence
  tokenizer instead of a regex.
- **PDF paragraph detection is unreliable**, as we discussed above -- this
  strategy works best when your source format preserves real structure (Word,
  HTML, Markdown) and is weaker on raw PDF text.

### When this comes up in interviews

A common follow-up after "why not fixed-size" is *"okay, so just split by
sentence then -- what's still wrong with that?"* The answer they're checking for:
sentence/paragraph chunking fixes the *grammar* problem but not the *meaning*
problem -- two adjacent, grammatically-clean chunks can still be about entirely
unrelated topics, or one coherent idea can still span multiple chunks if it's
expressed across several paragraphs. That's the motivation for semantic chunking
(Section 5).

---
✅ **Checkpoint — Section 2 complete.**
We now have `sentence_chunks_pdf` and `paragraph_chunks_docx`.
Confirm this looks right, and we'll move to **Section 3: Recursive Character chunking**.


## Section 3 — Recursive Character Chunking

**The idea in plain words:** this is the strategy almost every RAG tutorial
defaults to, and for good reason -- it's fixed-size chunking's smarter sibling.
Instead of always cutting by character count, it tries a *list of separators*,
biggest structural unit first, and only falls back to a smaller unit when it has
to.

### How it works (ASCII picture)

Typical separator list, tried in this order: `["\n\n", "\n", " ", ""]`

```
1. Try splitting on "\n\n" (paragraph breaks).
   Is each resulting piece small enough (<= chunk_size)?
      YES -> keep it as a chunk, done with that piece.
      NO  -> that piece is still too big, so:

2. Try splitting THAT piece on "\n" (line breaks).
      YES small enough -> keep it.
      NO  -> still too big, so:

3. Try splitting on " " (spaces / words).
      YES small enough -> keep it.
      NO -> still too big, so:

4. Fall back to "" -- plain fixed-size character cutting (Section 1's method),
   as the last resort, only on the pieces that truly couldn't be split any
   other way.
```

So it's not "one splitting rule" -- it's "try the nicest rule first, and only
degrade to a cruder rule where the nicer one doesn't produce small-enough
pieces." That's why it's called *recursive*: it recurses down the list of
separators, piece by piece.


### 3.1 Using LangChain's `RecursiveCharacterTextSplitter`

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],  # tried in this order, biggest unit first
)

recursive_chunks_pdf = recursive_splitter.split_text(pdf_text)
recursive_chunks_docx = recursive_splitter.split_text(docx_text)

print(f"PDF  -> {len(recursive_chunks_pdf)} chunks, sizes: {[len(c) for c in recursive_chunks_pdf[:8]]} ...")
print(f"DOCX -> {len(recursive_chunks_docx)} chunks, sizes: {[len(c) for c in recursive_chunks_docx[:8]]} ...")


PDF  -> 25 chunks, sizes: [448, 496, 423, 420, 328, 407, 467, 469] ...
DOCX -> 12 chunks, sizes: [495, 446, 475, 460, 281, 305, 401, 478] ...


In [13]:
for i, chunk in enumerate(recursive_chunks_pdf[:3]):
    print(f"--- PDF Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)
    print()


--- PDF Chunk 1 (448 chars) ---
[[PAGE:1]]
Clinical Trial Protocol
Protocol Number: CTP-2026-0142
Study Title: A Phase II, Randomized, Double-Blind, Placebo-Controlled Study of Oral Compound AX-119
in Adult Patients with Moderate-to-Severe Rheumatoid Arthritis
Sponsor: Meridian Biotherapeutics Inc.
Version: 3.1 | Date: 15-Jan-2026
1. Study Objectives
1.1 Primary Objective
The primary objective of this study is to evaluate the efficacy of AX-119 compared to placebo in reducing

--- PDF Chunk 2 (496 chars) ---
disease activity, as measured by the American College of Rheumatology 20% response criteria (ACR20),
at Week 24 in adult patients with moderate-to-severe rheumatoid arthritis who have had an inadequate
response to methotrexate.
1.2 Secondary Objectives

To assess the safety and tolerability of AX-119 over 24 weeks of treatment.

To evaluate the proportion of patients achieving ACR50 and ACR70 response at Week 24.

To assess change from baseline in the Disease Activity Score (DAS

**Compare to Section 1 (fixed-size) on the exact same `chunk_size=500,
overlap=50` settings.** Fixed-size cut straight through "disease" mid-word.
Here, because `RecursiveCharacterTextSplitter` prefers to cut on `"\n"` or
`" "` whenever it can, chunk boundaries fall on word or line breaks instead
of mid-word -- while still keeping to roughly the same target size.

Compare to Section 2 (sentence chunking): sentence chunking guarantees a cut
never happens inside a sentence, but chunk sizes end up uneven. Recursive
chunking gives you back tighter, more predictable chunk sizes (good when you
have a hard token budget) at the cost of occasionally still cutting inside a
sentence -- it only guarantees "not mid-word", not "not mid-sentence", unless
your separator list explicitly includes sentence-ending punctuation (notice
we added `". "` to the separator list above specifically to nudge it toward
sentence boundaries when possible).


### 3.2 Same splitter on the DOCX

In [14]:
for i, chunk in enumerate(recursive_chunks_docx[:3]):
    print(f"--- DOCX Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)
    print()


--- DOCX Chunk 1 (495 chars) ---
Clinical Data Handling and Privacy Policy
Document ID: POL-COMP-014   |   Version 2.3   |   Effective Date: 01-Feb-2026
1. Purpose
This policy establishes the requirements for the collection, storage, processing, and disclosure of clinical trial participant data across all studies sponsored or managed by Meridian Biotherapeutics Inc. It applies to all employees, contractors, clinical research organizations (CROs), and third-party vendors who handle participant data on behalf of the company.

--- DOCX Chunk 2 (446 chars) ---
2. Scope
This policy covers protected health information (PHI), personally identifiable information (PII), and any derived clinical or genomic data collected during the conduct of clinical trials, observational studies, and post-marketing surveillance activities. It applies regardless of the format in which the data is stored, including electronic health records, case report forms, biological samples, and imaging data.
3. Regulatory 

### Pros

- **The industry default for a reason:** good balance of simplicity, speed,
  and quality. No embeddings or model calls needed.
- **Respects structure when it can, degrades gracefully when it can't.** A
  huge, unbroken block of text (e.g. a giant table dumped as one paragraph)
  still gets split safely instead of becoming one enormous chunk.
- **Tunable** -- you control the separator list and can tailor it per document
  type (e.g. add `"```"` as a high-priority separator for chunking code, or
  Markdown heading markers for chunking `.md` files).
- **Fast and library-supported** -- LangChain, LlamaIndex, and most RAG
  frameworks ship this as the default splitter.

### Cons

- **Still not meaning-aware.** Like sentence/paragraph chunking, it has no idea
  whether two adjacent pieces of text are about the same topic.
- **Separator list needs tuning per document type** -- the defaults that work
  well for a blog post won't be ideal for a legal contract or a lab report
  full of tables and numbers.
- **Can still occasionally cut mid-sentence** if no earlier separator produced
  small-enough pieces.

### When this comes up in interviews

This is usually the answer to *"what would you actually use as your default
chunking strategy in a new RAG project?"* -- it's the pragmatic, well-tested
choice. A strong answer mentions that you'd *start* here, measure retrieval
quality, and only reach for something more expensive (semantic or structure-aware)
if this isn't good enough for your specific document type. It's also worth
knowing that this is what LangChain's own docs recommend as the default.

---
✅ **Checkpoint — Section 3 complete.**
We now have `recursive_chunks_pdf` and `recursive_chunks_docx`.
Confirm this looks right, and we'll move to **Section 4: Structure-aware chunking**.


## Section 4 — Structure-aware Chunking

**The idea in plain words:** use the document's own structure -- headings,
numbered sections, bullet lists -- as the chunk boundaries, instead of
character counts or sentence counts. A chunk becomes "everything under
Heading X, until the next heading."

### How it works (ASCII picture)

```
1. Purpose
   <body text about purpose...>          \
                                           |--> Chunk 1: "1. Purpose" + its body
2. Scope                                 /
   <body text about scope...>            \
                                           |--> Chunk 2: "2. Scope" + its body
3. Regulatory Framework                  /
   <body text + bullet list...>          \
                                           |--> Chunk 3: "3. Regulatory Framework" + its body
```

Each chunk is a **complete, self-contained section** of the document -- not a
character count, not a sentence count, but a *document unit that the author
already intended to be read together*.

This strategy needs structure to detect in the first place, so we'll use two
different techniques:
- **DOCX** already tells us which paragraphs are headings (`Heading 1` style)
  -- we saw this in Section 0. We just group on that directly.
- **PDF** lost that formatting metadata during text extraction (pypdf gives us
  plain text, not "this line was Heading 1"). So we detect headings with a
  pattern instead: our protocol's headings look like `"7. Safety Assessments"`
  or `"8.2 Statistical Analysis"` -- a number, then a title. We match that pattern.


### 4.1 Structure-aware chunking on the DOCX (using real heading styles)

In [15]:
def structure_aware_chunks_docx(paragraphs):
    """paragraphs: list of (style, text). Start a new chunk every time we hit
    a Heading style; everything until the next heading belongs to that chunk.
    """
    chunks = []
    current_heading = None
    current_body = []

    def flush():
        if current_heading is not None or current_body:
            heading_line = f"{current_heading}\n" if current_heading else ""
            chunks.append((current_heading, heading_line + "\n".join(current_body)))

    for style, text in paragraphs:
        if style.startswith("Heading"):
            flush()
            current_heading = text
            current_body = []
        else:
            current_body.append(text)
    flush()
    return chunks


structure_chunks_docx = structure_aware_chunks_docx(docx_paragraphs)

print(f"Number of chunks: {len(structure_chunks_docx)}")
print()
for heading, chunk in structure_chunks_docx[:4]:
    print(f"=== SECTION: {heading} ({len(chunk)} chars) ===")
    print(chunk)
    print()


Number of chunks: 14

=== SECTION: None (119 chars) ===
Clinical Data Handling and Privacy Policy
Document ID: POL-COMP-014   |   Version 2.3   |   Effective Date: 01-Feb-2026

=== SECTION: 1. Purpose (375 chars) ===
1. Purpose
This policy establishes the requirements for the collection, storage, processing, and disclosure of clinical trial participant data across all studies sponsored or managed by Meridian Biotherapeutics Inc. It applies to all employees, contractors, clinical research organizations (CROs), and third-party vendors who handle participant data on behalf of the company.

=== SECTION: 2. Scope (422 chars) ===
2. Scope
This policy covers protected health information (PHI), personally identifiable information (PII), and any derived clinical or genomic data collected during the conduct of clinical trials, observational studies, and post-marketing surveillance activities. It applies regardless of the format in which the data is stored, including electronic health records, ca

### 4.2 Structure-aware chunking on the PDF (using a heading pattern)

In [16]:
import re

# Matches only TOP-LEVEL numbered headings like "1. Study Objectives" or
# "13. Appendix: Schedule of Assessments (Summary)" -- deliberately does NOT
# match sub-headings like "1.1 Primary Objective" or "4.2 Exclusion Criteria",
# so subsections stay merged inside their parent section's chunk.
HEADING_PATTERN = re.compile(r"^\d+\.\s+[A-Z][A-Za-z].*$")

def structure_aware_chunks_pdf(text: str):
    lines = [ln.strip() for ln in text.split("\n") if ln.strip()]
    chunks = []
    current_heading = None
    current_body = []

    def flush():
        if current_heading is not None or current_body:
            heading_line = f"{current_heading}\n" if current_heading else ""
            chunks.append((current_heading, heading_line + " ".join(current_body)))

    for line in lines:
        if line.startswith("[[PAGE:"):
            continue  # skip our own page markers, not real content
        if HEADING_PATTERN.match(line):
            flush()
            current_heading = line
            current_body = []
        else:
            current_body.append(line)
    flush()
    return chunks


structure_chunks_pdf = structure_aware_chunks_pdf(pdf_text)

print(f"Number of chunks: {len(structure_chunks_pdf)}")
print()
for heading, chunk in structure_chunks_pdf[:4]:
    print(f"=== SECTION: {heading} ({len(chunk)} chars) ===")
    print(chunk)
    print()


Number of chunks: 14

=== SECTION: None (289 chars) ===
Clinical Trial Protocol Protocol Number: CTP-2026-0142 Study Title: A Phase II, Randomized, Double-Blind, Placebo-Controlled Study of Oral Compound AX-119 in Adult Patients with Moderate-to-Severe Rheumatoid Arthritis Sponsor: Meridian Biotherapeutics Inc. Version: 3.1 | Date: 15-Jan-2026

=== SECTION: 1. Study Objectives (734 chars) ===
1. Study Objectives
1.1 Primary Objective The primary objective of this study is to evaluate the efficacy of AX-119 compared to placebo in reducing disease activity, as measured by the American College of Rheumatology 20% response criteria (ACR20), at Week 24 in adult patients with moderate-to-severe rheumatoid arthritis who have had an inadequate response to methotrexate. 1.2 Secondary Objectives  To assess the safety and tolerability of AX-119 over 24 weeks of treatment.  To evaluate the proportion of patients achieving ACR50 and ACR70 response at Week 24.  To assess change from baseline in t

**Compare this to every previous strategy.** Notice `structure_chunks_pdf` now
holds complete sections -- "1. Study Objectives" (both 1.1 Primary and 1.2
Secondary objectives together) is one self-contained chunk, not sliced across
2-3 arbitrary chunk boundaries like it was in Sections 1-3. If someone asks
*"what are the secondary objectives of this study?"*, this chunk contains the
whole, complete answer -- no missing context from a neighboring chunk.

The trade-off is visible too: chunk sizes are now wildly uneven (compare the
short "6. Efficacy Assessments" heading-only chunks to a long combined
section) -- some sections are naturally longer than others, and this strategy
doesn't try to normalize that.

### Pros

- **Chunks map to what a human would call "a topic."** This is usually the
  *best* strategy for retrieval quality when your document actually has
  strong structure (protocols, contracts, policies, technical manuals, SOPs).
- **No embeddings or model calls needed** -- still just pattern matching.
- **Preserves the heading as context**, which is valuable metadata: you can
  show "Section 8.2: Statistical Analysis" as a citation label to the user,
  and you can filter/boost retrieval by section type.

### Cons

- **Completely dependent on the document having detectable structure.** Works
  great on our protocol and policy; would do nothing useful on an unstructured
  free-text email or a transcript.
- **Requires format-specific extraction logic.** We needed two different
  approaches for DOCX (style metadata) vs. PDF (regex pattern) -- and the PDF
  regex is fragile; it would break on a differently-formatted protocol.
- **Uneven, sometimes very large chunks.** A long section can still blow past
  a token budget -- in production this strategy is usually combined with
  recursive/fixed-size chunking as a second pass *inside* any section that's
  still too big (a "structure first, then recursive within each section"
  pipeline).
- **Doesn't handle content that spans sections** -- e.g. a table that
  continues across a section break.

### When this comes up in interviews

This is the answer to *"you're building RAG over [contracts / SOPs / medical
protocols / policies] -- what would you actually do differently from a generic
blog-post chunker?"* Strong answers mention: detect and use the document's own
structure first, and only fall back to size-based splitting inside sections that
are still too large. This is exactly the kind of "know your document type"
thinking that separates a junior implementation from a senior one.

---
✅ **Checkpoint — Section 4 complete.**
We now have `structure_chunks_pdf` and `structure_chunks_docx` (each a list of `(heading, text)` tuples).
Confirm this looks right, and we'll move to **Section 5: Semantic chunking**.


## Section 5 — Semantic Chunking

**The idea in plain words:** instead of using grammar rules (sentence/paragraph)
or document structure (headings), actually measure how *similar in meaning*
consecutive sentences are, and only cut where the meaning clearly shifts.

### How it works (ASCII picture)

```
Sentence:        S1   S2   S3   S4   S5   S6   S7   S8
Similarity to
previous sentence: -  0.91 0.88 0.85 0.30 0.90 0.87 0.28
                              ^low^                ^low^
                         (topic shift)        (topic shift)

Chunks:          [ S1  S2  S3  S4 ] [ S5  S6  S7 ] [ S8 ...
                   still about topic A   topic B      topic C
```

1. Split the text into sentences.
2. Turn each sentence into a vector (an "embedding") that represents its meaning.
3. Compute the similarity between each sentence and the one right before it.
4. Wherever similarity drops sharply (a "topic shift"), that's a chunk boundary.
5. Group sentences between boundaries into chunks.

**Important honest note about what we're using for step 2:** a real production
system would use a neural embedding model (Amazon Titan Embeddings -- which you're
already using in your multi-tenant platform ingestion pipeline -- OpenAI, Cohere,
or `sentence-transformers`) which actually understands meaning, synonyms, and
context. Those models need to download weights from the internet or call a paid
API. To keep this notebook runnable **completely offline, by anyone, with no API
key**, we'll use **TF-IDF vectors** (`scikit-learn`) instead -- these represent a
sentence by *which words it uses*, not true meaning. It's a legitimate, classic
technique, but it's a meaning proxy, not real semantic understanding (it won't
know that "physician" and "doctor" mean the same thing, for example). We call
this out explicitly in the pros/cons below.


### 5.1 Semantic chunking on the PDF

In [20]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def semantic_chunks(text: str, similarity_drop_percentile: int = 25, min_sentences_per_chunk: int = 1):
    """Embed each sentence with TF-IDF, measure similarity to the previous
    sentence, and cut wherever that similarity falls in the bottom
    `similarity_drop_percentile` of all observed similarities (i.e. cut at the
    weakest connections in the document, not at a fixed threshold -- this
    adapts to each document instead of needing a hand-tuned cutoff number).
    """
    sentences = split_into_sentences(text)
    if len(sentences) < 3:
        return [text]

    vectorizer = TfidfVectorizer(stop_words="english")
    vectors = vectorizer.fit_transform(sentences)

    sims = []
    for i in range(1, len(sentences)):
        sim = cosine_similarity(vectors[i - 1], vectors[i])[0][0]
        sims.append(sim)

    cutoff = np.percentile(sims, similarity_drop_percentile)

    chunks, current = [], [sentences[0]]
    for i, sim in enumerate(sims):
        next_sentence = sentences[i + 1]
        if sim <= cutoff and len(current) >= min_sentences_per_chunk:
            chunks.append(" ".join(current))
            current = [next_sentence]
        else:
            current.append(next_sentence)
    if current:
        chunks.append(" ".join(current))
    return chunks


semantic_chunks_pdf = semantic_chunks(pdf_text, similarity_drop_percentile=25)

print(f"Number of chunks: {len(semantic_chunks_pdf)}")
print(f"Chunk sizes (chars): {[len(c) for c in semantic_chunks_pdf[:8]]} ...")
print()
for i, chunk in enumerate(semantic_chunks_pdf[:3]):
    print(f"--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)
    print()


NameError: name 'split_into_sentences' is not defined

### 5.2 Same strategy on the DOCX

In [18]:
semantic_chunks_docx = semantic_chunks(docx_text, similarity_drop_percentile=25)

print(f"Number of chunks: {len(semantic_chunks_docx)}")
print()
for i, chunk in enumerate(semantic_chunks_docx[:3]):
    print(f"--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)
    print()


Number of chunks: 8

--- Chunk 1 (751 chars) ---
Clinical Data Handling and Privacy Policy Document ID: POL-COMP-014 | Version 2.3 | Effective Date: 01-Feb-2026 1. Purpose This policy establishes the requirements for the collection, storage, processing, and disclosure of clinical trial participant data across all studies sponsored or managed by Meridian Biotherapeutics Inc. It applies to all employees, contractors, clinical research organizations (CROs), and third-party vendors who handle participant data on behalf of the company. 2. Scope This policy covers protected health information (PHI), personally identifiable information (PII), and any derived clinical or genomic data collected during the conduct of clinical trials, observational studies, and post-marketing surveillance activities.

--- Chunk 2 (732 chars) ---
It applies regardless of the format in which the data is stored, including electronic health records, case report forms, biological samples, and imaging data. 3. Regulato

**Compare to structure-aware (Section 4).** Structure-aware chunking used the
*author's* explicit signal (a numbered heading) to decide where sections begin
and end. Semantic chunking doesn't need the author to have marked anything --
it discovers topic shifts purely from the text itself. That matters a lot for
documents that *don't* have clean headings (meeting transcripts, long emails,
free-text clinical notes) where structure-aware chunking has nothing to grab
onto, but semantic chunking still works.

The trade-off: with our TF-IDF proxy, "similar meaning" really means "shares a
lot of the same distinctive words." Two sentences that are genuinely about the
same medical concept but phrased with different vocabulary might get flagged
as a false topic shift. A real embedding model handles that correctly; TF-IDF
sometimes won't.

### Pros

- **Doesn't require the document to have any explicit structure** -- works on
  free text, transcripts, notes, anything.
- **Adapts per-document** -- the percentile-based cutoff finds *this* document's
  weakest connections, rather than using one fixed similarity number for every
  document (which would be too strict for some documents and too loose for others).
- **With a real embedding model** (not our TF-IDF stand-in), this captures actual
  paraphrase and synonym relationships that grammar-based or structure-based
  approaches simply cannot see.

### Cons

- **Needs an embedding step for every sentence**, which costs time and (with a
  real model) money or GPU/API calls -- much more expensive than any strategy
  so far.
- **Quality is only as good as the embedding model.** Our offline TF-IDF version
  is a teaching stand-in, not production-grade; production needs a real
  embedding model, which reintroduces a network/API dependency.
- **The similarity threshold is a knob you have to tune** -- too strict and you
  get one chunk per sentence (over-fragmented); too loose and you get one giant
  chunk (no split at all).
- **Slower** than every strategy before it, especially on large document sets.

### When this comes up in interviews

This is the answer to *"how would you chunk unstructured content -- like call
transcripts or long-form free-text clinical notes -- where there's no heading to
anchor on?"* The expected answer: semantic chunking, using real sentence
embeddings, with an adaptive threshold (percentile-based, like we did, or a
fixed cosine-distance cutoff tuned on a validation set). A senior-level answer
also flags the cost trade-off: this is meaningfully more expensive to run at
scale than structural methods, so it's usually reserved for content that truly
lacks structure, not applied blindly everywhere.

---
✅ **Checkpoint — Section 5 complete.**
We now have `semantic_chunks_pdf` and `semantic_chunks_docx`.
Confirm this looks right, and we'll move to **Section 6: LLM / Agentic chunking**.


## Section 6 — LLM / Agentic Chunking

**The idea in plain words:** use an actual LLM to make chunking decisions,
instead of rules, regex, or similarity math. The LLM reads a piece of text and
either (a) decides where the natural boundaries are, or (b) takes a chunk that
some other, cheaper strategy already produced and improves it -- rewriting it
so it stands alone and makes sense without the rest of the document around it.

This is called **"agentic"** because the LLM isn't just doing one fixed
transformation -- it's making a judgment call, sometimes in a loop: read,
evaluate, decide, and (in more advanced setups) even revise its own decision.

### How it works (ASCII picture)

```
Cheap strategy (e.g. structure-aware, Section 4)
        |
        v
  [ Raw chunk: "This policy covers PHI, PII, and derived clinical or        ]
  [ genomic data collected during trials..." ]        <- reads fine WITH
                                                           the doc around it,
                                                           but alone, it's
                                                           missing "which
                                                           policy? whose data?"
        |
        v  LLM reads the chunk + a little surrounding context
        v
  [ Contextualized chunk: "This excerpt is from Meridian Biotherapeutics'    ]
  [ Clinical Data Handling and Privacy Policy (Section 2, Scope). It states  ]
  [ that the policy covers PHI, PII, and derived clinical or genomic data... ]
```

This specific pattern -- take a normal chunk, then have an LLM prepend a short
piece of context so the chunk is understandable on its own -- is close to what
Anthropic calls **"Contextual Retrieval"**: cheap structural/recursive chunking
does the heavy lifting of splitting the document, and the LLM's one job is to
make each resulting chunk self-contained. This is usually far more practical
than asking an LLM to decide every chunk boundary from scratch, because reading
an entire long document through an LLM purely to decide *where to cut* is slow
and expensive at scale -- better to spend the LLM's effort on the part that
actually needs judgment (does this chunk make sense alone?), not on the part a
regex already does well (finding a paragraph break).


### 6.1 Implementation: LLM-generated context per chunk

In [19]:
import os

def mock_llm_context(chunk_text: str, heading) -> str:
    """Stand-in for an LLM call, used when no API key is configured, so this
    notebook still runs end-to-end for anyone without credentials. It's a
    simple heuristic, NOT a real understanding of the text -- just enough to
    show the shape of what an LLM call would produce.
    """
    heading_text = heading if heading else "the opening section"
    first_sentence = split_into_sentences(chunk_text)[0] if chunk_text.strip() else ""
    return (f"This excerpt is from the section '{heading_text}' of the document. "
            f"It begins: \"{first_sentence[:100]}\"")


def llm_context(chunk_text: str, heading, document_title: str) -> str:
    """Real implementation: ask an LLM to write a short (1-2 sentence) piece
    of context for this chunk, given the chunk and which document/section it's
    from. Requires ANTHROPIC_API_KEY to be set as an environment variable --
    falls back to the mock above if it isn't, so the notebook always runs.
    """
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        return mock_llm_context(chunk_text, heading)

    import anthropic
    client = anthropic.Anthropic(api_key=api_key)

    prompt = f"""Document: {document_title}
Section heading: {heading or "(no heading)"}

Chunk text:
\"\"\"
{chunk_text[:1500]}
\"\"\"

Write ONE short sentence of context (max 25 words) that helps this chunk be
understood on its own, if read completely out of context from the rest of the
document. Do not summarize the whole chunk -- just give the reader the "where
am I / what is this part of" context. Reply with only that one sentence."""

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=100,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text.strip()


print("ANTHROPIC_API_KEY set:", bool(os.environ.get("ANTHROPIC_API_KEY")))
print("(If not set, we'll use the mock function below so the notebook still runs.")
print(" To use the real thing: os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...' before this cell.)")


ANTHROPIC_API_KEY set: False
(If not set, we'll use the mock function below so the notebook still runs.
 To use the real thing: os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...' before this cell.)


In [20]:
# Build agentic chunks: take our structure-aware chunks (Section 4) as the base
# -- cheap, reliable splitting already done -- and add LLM-generated context
# to each one. This is the "agentic" step: a judgment call per chunk.

agentic_chunks_pdf = []
for heading, chunk_text in structure_chunks_pdf:
    context = llm_context(chunk_text, heading, document_title="Clinical Trial Protocol CTP-2026-0142")
    agentic_chunks_pdf.append(f"[Context: {context}]\n{chunk_text}")

print(f"Number of chunks: {len(agentic_chunks_pdf)}")
print()
for chunk in agentic_chunks_pdf[:2]:
    print("---")
    print(chunk[:500])
    print()


Number of chunks: 14

---
[Context: This excerpt is from the section 'the opening section' of the document. It begins: "Clinical Trial Protocol Protocol Number: CTP-2026-0142 Study Title: A Phase II, Randomized, Double-B"]
Clinical Trial Protocol Protocol Number: CTP-2026-0142 Study Title: A Phase II, Randomized, Double-Blind, Placebo-Controlled Study of Oral Compound AX-119 in Adult Patients with Moderate-to-Severe Rheumatoid Arthritis Sponsor: Meridian Biotherapeutics Inc. Version: 3.1 | Date: 15-Jan-2026

---
[Context: This excerpt is from the section '1. Study Objectives' of the document. It begins: "1."]
1. Study Objectives
1.1 Primary Objective The primary objective of this study is to evaluate the efficacy of AX-119 compared to placebo in reducing disease activity, as measured by the American College of Rheumatology 20% response criteria (ACR20), at Week 24 in adult patients with moderate-to-severe rheumatoid arthritis who have had an inadequate response to methotrexate. 1.2 Sec

**Compare to structure-aware chunking (Section 4).** The underlying text and
boundaries are identical -- we deliberately reused `structure_chunks_pdf` as the
base, because letting an LLM decide *where* to cut a well-structured document
would be solving a problem regex already solved for free. What's new is the
`[Context: ...]` line at the top of each chunk. That one line is what actually
moves the needle on retrieval: when this chunk gets embedded and searched
later, the embedding now also reflects "this is about the clinical trial
protocol's efficacy assessments," not just the raw paragraph text -- which
helps a lot when the chunk itself is full of pronouns, abbreviations, or
implicit references ("this study," "the sponsor," "the primary endpoint")
that only make sense with the missing context restored.

### A second common agentic pattern (concept only, not implemented here)

Instead of context-labeling existing chunks, some pipelines let an LLM
**propose the boundaries directly**, especially for messy, unstructured text
(scanned notes, meeting transcripts). A simplified version of that prompt:

```
"Here is a document. Split it into self-contained sections, each covering
one topic. Return each section as a JSON object with a short title and the
full text. A section should be understandable without reading the others."
```

We don't run this here because doing that over an entire real document is
comparatively expensive (large context, multiple LLM calls or one very large
call) and non-deterministic (ask twice, may get slightly different boundaries)
-- worth knowing about, but the context-augmentation pattern above is the one
you're more likely to actually deploy.

### Pros

- **Handles cases nothing else can.** An LLM can recognize when a chunk needs
  outside context to make sense in a way no regex or similarity score can.
- **Directly improves retrieval on ambiguous, pronoun-heavy, or acronym-heavy
  text** -- extremely common in clinical and compliance documents ("the
  Sponsor," "the IRB," "this policy") where a chunk read in isolation is
  genuinely confusing without the LLM-added context.
- **Composable** -- it's a *layer on top of* any earlier strategy, not a
  replacement. You get to keep the cheap, reliable boundaries from Section 3
  or 4 and only pay the LLM cost for the judgment call.

### Cons

- **The most expensive strategy by far** -- one LLM call per chunk, at minimum.
  On a large document set, this is a real, ongoing cost, not a one-time cost.
- **Slowest** -- network round-trips per chunk, unless batched carefully.
- **Non-deterministic** -- ask the same chunk twice, might get a slightly
  different context sentence. Needs to be an accepted trade-off, not a surprise.
- **Requires infrastructure**: API keys, rate limit handling, retries, cost
  monitoring -- meaningfully more operational complexity than every prior
  strategy, all of which needed zero external dependencies at chunk-time.

### When this comes up in interviews

This is the answer to *"the retrieval is technically finding the right chunk,
but the LLM's final answer is still wrong or confused -- what would you check?"*
A strong answer: check whether the retrieved chunk is *self-contained*. If it's
full of unresolved pronouns or references to "the above" that don't survive
being pulled out of context, that's exactly what LLM-based context augmentation
(or Anthropic's Contextual Retrieval approach) is built to fix -- and it's a
strong signal of "I know what happens after chunking, not just chunking itself."

---
✅ **Checkpoint — Section 6 complete.**
We now have `agentic_chunks_pdf`. Every one of the six strategies is done:
`fixed_chunks_pdf`, `sentence_chunks_pdf`, `recursive_chunks_pdf`, `structure_chunks_pdf`,
`semantic_chunks_pdf`, `agentic_chunks_pdf`.
Confirm this looks right, and we'll move to **Section 7: building a vector store
and comparing retrieval quality across all six strategies.**


## Section 7 — Vector Store + Retrieval Comparison

Now the part that actually matters: **does the chunking strategy change what
gets retrieved when someone asks a real question?**

We'll build one small vector store per strategy (all six, on the clinical
trial protocol), ask the same 5 questions against every one, and look at what
each strategy's top result actually contains.

### How a vector store works (ASCII picture)

```
Chunks:  [chunk 1] [chunk 2] [chunk 3] ... [chunk N]
             |         |         |             |
             v         v         v             v
         embed()   embed()   embed()       embed()
             |         |         |             |
             v         v         v             v
         [vec 1]   [vec 2]   [vec 3]   ...  [vec N]   <-- stored in the index

Question: "What is the primary endpoint?"
             |
             v
          embed()
             |
             v
        [query vec]  --> compare to every stored vector (cosine similarity)
                     --> return the chunk(s) whose vector is closest
```

**Same honest note as Section 5:** a production system embeds with a real model
(Amazon Titan Embeddings, OpenAI, Bedrock, `sentence-transformers`). To keep
this notebook runnable offline with no API key, we use TF-IDF vectors again as
our embedding stand-in -- and we use **FAISS** (`faiss-cpu`) as the actual
vector index, exactly like production, so this part transfers directly: swap
in a real embedding function later and everything else here stays the same.

This also mirrors the same cost-first choice from your own multi-tenant
platform -- Phase 0 there deliberately uses brute-force cosine similarity over
a small JSON file in S3 instead of standing up a managed vector DB service,
because a managed vector DB is unnecessary spend at small scale. FAISS's
`IndexFlatIP` (used below) is doing exactly that brute-force comparison, just
running locally instead of against an S3 file -- same underlying idea, just
swapped storage.


### 7.1 A small reusable vector store class

In [21]:
import faiss
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize


class SimpleVectorStore:
    """Wraps a TF-IDF vectorizer + a FAISS index. In production you would
    swap `TfidfVectorizer` for a call to a real embedding model (Titan,
    OpenAI, sentence-transformers) -- everything else here (the FAISS index,
    the search logic) stays exactly the same.
    """

    def __init__(self, chunks):
        self.chunks = chunks
        self.vectorizer = TfidfVectorizer(stop_words="english")
        vectors = self.vectorizer.fit_transform(chunks).toarray().astype("float32")
        vectors = normalize(vectors)  # so inner product == cosine similarity
        self.dim = vectors.shape[1]
        self.index = faiss.IndexFlatIP(self.dim)  # IP = inner product
        self.index.add(vectors)

    def search(self, query: str, k: int = 1):
        query_vec = self.vectorizer.transform([query]).toarray().astype("float32")
        query_vec = normalize(query_vec)
        scores, indices = self.index.search(query_vec, k)
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:
                continue
            results.append((float(score), self.chunks[idx]))
        return results


# Build one vector store per strategy, all on the PDF (our richer document)
stores = {
    "1. Fixed-size":       SimpleVectorStore(fixed_chunks_pdf),
    "2. Sentence":         SimpleVectorStore(sentence_chunks_pdf),
    "3. Recursive":        SimpleVectorStore(recursive_chunks_pdf),
    "4. Structure-aware":  SimpleVectorStore([text for _, text in structure_chunks_pdf]),
    "5. Semantic":         SimpleVectorStore(semantic_chunks_pdf),
    "6. Agentic":          SimpleVectorStore(agentic_chunks_pdf),
}

for name, store in stores.items():
    print(f"{name:22s} -> {len(store.chunks):3d} chunks indexed")


1. Fixed-size          ->  24 chunks indexed
2. Sentence            ->  24 chunks indexed
3. Recursive           ->  25 chunks indexed
4. Structure-aware     ->  14 chunks indexed
5. Semantic            ->  13 chunks indexed
6. Agentic             ->  14 chunks indexed


### 7.2 Five questions, run against every strategy

In [22]:
questions = [
    "What is the primary efficacy endpoint of this study?",
    "What are the key exclusion criteria for study participation?",
    "How many patients will be enrolled and how are they randomized?",
    "Who reviews the unblinded safety data during the study?",
    "What is required before a patient can be enrolled, in terms of consent and ethics approval?",
]

import textwrap

for q_num, question in enumerate(questions, start=1):
    print("=" * 90)
    print(f"Q{q_num}: {question}")
    print("=" * 90)
    for strategy_name, store in stores.items():
        results = store.search(question, k=1)
        score, chunk = results[0]
        preview = textwrap.shorten(chunk.replace("\n", " "), width=180)
        print(f"[{strategy_name:22s}] score={score:.3f}  {preview}")
    print()


Q1: What is the primary efficacy endpoint of this study?
[1. Fixed-size         ] score=0.303  ic diary. 5.2 Concomitant Medications Patients must continue background methotrexate at a stable dose throughout the study. Stable doses of oral corticosteroids (equivalent [...]
[2. Sentence           ] score=0.320  Stable doses of oral corticosteroids (equivalent to prednisone 10 mg/day or less) and non-steroidal anti-inflammatory drugs (NSAIDs) are permitted. Live vaccines are [...]
[3. Recursive          ] score=0.288  oral corticosteroids (equivalent to prednisone 10 mg/day or less) and non-steroidal anti-inflammatory drugs (NSAIDs) are permitted. Live vaccines are prohibited from [...]
[4. Structure-aware    ] score=0.206  6. Efficacy Assessments The primary efficacy endpoint, ACR20 response at Week 24, will be assessed by a blinded joint assessor at each scheduled visit. Tender and swollen [...]
[5. Semantic           ] score=0.198  Efficacy Assessments The primary efficacy endpoint, A

**How to read this table:** for each question, look at which strategies
returned a chunk that actually *contains the answer*, versus a chunk that's
merely on-topic but missing the specific detail.

Here's what actually happened when this ran (your exact numbers may vary
slightly by environment, but the patterns will hold) -- and it's more
interesting than a clean "strategy X always wins" story:

- **Q1 (primary efficacy endpoint):** Structure-aware, Semantic, and Agentic
  all retrieved the exact right chunk (Section 6, containing "The primary
  efficacy endpoint, ACR20 response at Week 24..."). Fixed-size, Sentence, and
  Recursive all missed it, returning a chunk about concomitant medications
  instead -- likely because their chunk boundaries split the efficacy section
  awkwardly, weakening its TF-IDF match relative to a less relevant chunk.
- **Q2 (exclusion criteria) -- every single strategy got this wrong.** All six
  returned a chunk about *withdrawal/discontinuation* criteria, not the actual
  exclusion criteria (Section 4.2). This is a genuine, honest failure, and a
  great example of a real limitation: "exclusion" and "withdrawal" share
  enough vocabulary (patients, criteria, study, discontinue) that our TF-IDF
  proxy treats them as similar, even though they mean very different things.
  A real embedding model, trained to understand meaning rather than word
  overlap, would very likely have told these apart. **This is exactly the
  kind of failure that motivates using real semantic embeddings in
  production, not TF-IDF** -- we kept TF-IDF for offline reproducibility, but
  this result is a live demonstration of its cost.
- **Q3 (enrollment/randomization):** Recursive and Structure-aware both found
  the right chunk (Section 3.1, "Approximately 240 patients will be
  randomized..."); the others latched onto the title page instead.
- **Q4 (who reviews safety data):** every strategy returned something
  relevant, but **Semantic scored notably highest (0.572 vs ~0.37 for the
  rest)** and returned the tightest, most direct answer -- a single sentence
  naming the DSMB -- rather than a whole section. This is semantic chunking's
  strength showing up concretely: because it cuts at meaning boundaries
  rather than heading boundaries, it can produce a small, precise chunk that's
  *just* the answer, when structure-aware is stuck returning the whole section
  the answer lives in.
- **Q5 (consent and ethics approval):** same pattern -- Semantic again
  returned the single sentence naming the IRB/IEC and informed consent
  requirement directly, while the others returned the broader "9. Ethical
  Considerations" section, which does contain the answer but makes the reader
  (or a downstream LLM) work harder to find it.

**The honest takeaway:** no single strategy won every question. Structure-aware
and Semantic were the strongest overall here, but for different reasons --
structure-aware wins when the question maps to a section the author already
delineated; semantic wins when the useful information is a specific sentence
buried inside a larger section. And Q2 is a genuine reminder that even a
"working" pipeline can retrieve confidently wrong chunks when the underlying
embedding is weak -- confidence (the similarity score) is not the same thing
as correctness.

**Production note:** this is a toy evaluation with 5 hand-picked questions and
`k=1`. A real evaluation would use a labeled test set of (question, correct
chunk) pairs, measure `k=3` or `k=5` (not just top-1), and report standard
retrieval metrics like Recall@k and Mean Reciprocal Rank (MRR) -- eyeballing
results like we just did is a fine first pass, not a substitute for that.

---
✅ **Checkpoint — Section 7 complete.**
We now have a searchable `stores` dict for all six strategies, and we've seen
real retrieval differences on real questions -- including a real failure case.
Confirm this looks right, and we'll move to **Section 8: the final comparison
table and healthcare-specific recommendations.**


## Section 8 — Final Comparison and Healthcare-Specific Recommendations

### Summary table

| Strategy | Respects sentences? | Respects structure? | Understands meaning? | Cost | Best for |
|---|---|---|---|---|---|
| **1. Fixed-size** | ❌ | ❌ | ❌ | 💲 (cheapest) | Hard token budgets; quick prototypes; last-resort fallback inside other strategies |
| **2. Sentence/Paragraph** | ✅ | Partial (paragraphs only) | ❌ | 💲 | Clean prose with little structure; quick quality win over fixed-size |
| **3. Recursive Character** | Mostly | Partial | ❌ | 💲 | **General-purpose default.** Start here for most projects. |
| **4. Structure-aware** | ✅ | ✅ | ❌ | 💲 (cheap, but needs structure-detection logic) | Contracts, SOPs, protocols, policies -- any document the author already organized into sections |
| **5. Semantic** | ✅ | N/A (ignores structure, uses meaning) | ✅ (with a real embedding model) | 💲💲 (embeddings) | Unstructured free text: transcripts, notes, emails, anything without headings |
| **6. LLM/Agentic** | ✅ | ✅ (inherits from base strategy) | ✅✅ (adds explicit reasoning) | 💲💲💲 (most expensive) | High-stakes retrieval where chunks must be self-contained (ambiguous pronouns, acronyms); usually layered on top of Structure-aware or Recursive, not used alone |

### A simple decision flow

```
Does the document have clear structure (headings, numbered sections)?
  YES -> Structure-aware chunking (Section 4)
         -> if any individual section is still too big for your token
            budget, apply Recursive Character chunking (Section 3) INSIDE
            that section as a second pass
  NO  -> Does retrieval quality on Recursive Character chunking (Section 3)
         already meet your bar?
           YES -> ship it, you likely don't need anything fancier
           NO  -> try Semantic chunking (Section 5) with a REAL embedding
                  model (not our TF-IDF stand-in)

Regardless of which strategy you land on:
  Are chunks full of unresolved pronouns/acronyms that only make sense
  with surrounding context (very common in clinical/compliance text --
  "the Sponsor," "the IRB," "this policy")?
    YES -> layer LLM/Agentic context augmentation (Section 6) on top
    NO  -> you're done
```

### Healthcare and compliance documents specifically

Every document we chunked in this notebook *was* a healthcare/compliance
document on purpose, so this isn't abstract -- here's what we actually saw:

1. **Structure-aware chunking should usually be your starting point**, not an
   afterthought, for this domain. Clinical protocols, SOPs, IRB submissions,
   compliance policies, and case report forms are almost always written with
   deliberate numbered sections *because regulators expect that structure* --
   which means the document is already telling you where the meaningful
   boundaries are. We saw this directly: Section 6 (Efficacy Assessments) and
   Section 9 (Ethical Considerations) retrieved cleanly and completely as
   whole, self-contained units.

2. **Watch out for near-synonym confusion between clinical terms** -- Q2 in
   Section 7 showed every single strategy confusing "exclusion criteria" with
   "withdrawal/discontinuation criteria." In a clinical or compliance setting,
   that's not a cosmetic error -- confusing eligibility criteria with
   withdrawal criteria in a real system could surface the wrong information to
   a clinician or a compliance officer at the wrong moment. This is a strong
   argument for (a) using a real, meaning-aware embedding model rather than
   TF-IDF in production, and (b) evaluating retrieval specifically on
   easily-confused near-synonym pairs common in your domain, not just on easy
   questions.

3. **Chunks in this domain are read by both people and downstream LLMs, and
   both need context.** A compliance officer scanning a retrieved snippet
   needs to know *which* policy and section it's from. An LLM generating an
   answer needs the same thing to avoid hallucinating specifics. This is
   exactly why the Contextual/Agentic layer (Section 6) is worth its extra
   cost specifically in this domain, even though it's the most expensive
   strategy -- the cost of a wrong or unclear answer in a clinical or
   compliance context is much higher than in, say, a general-knowledge chatbot.

4. **Never let a chunk boundary silently fall inside a table, a dosage, or a
   list of inclusion/exclusion criteria.** We didn't stress-test tables in
   this notebook, but it's worth flagging: PDF table extraction is its own
   hard problem (Section 0's production note about extraction quality applies
   double here), and a chunker that isn't table-aware can easily separate a
   drug name from its dosage, which is about as high-stakes a chunking failure
   as exists in this domain.

5. **For your Clinical Intelligence Platform specifically:** given the four
   personas you're building for (Researchers, Doctors, Compliance Officers,
   Clinical Operations), it's worth considering that different personas may
   genuinely want different chunking behavior over the *same* source
   documents -- a Compliance Officer benefits most from structure-aware
   chunks that map cleanly to policy sections they can cite, while a
   Researcher asking open-ended questions across many documents may get more
   value from semantic chunking that surfaces relevant passages regardless of
   which section they happen to sit in. That's a real architectural decision,
   not just a chunking detail -- it may be worth chunking the same source
   document more than one way and letting retrieval choose per query type,
   once you get to that phase of the roadmap.

---
## What we built, end to end

- Loaded and extracted text from a real PDF and a real DOCX
- Implemented **6 chunking strategies from scratch or with production libraries**:
  fixed-size, sentence/paragraph, recursive character, structure-aware, semantic, and LLM/agentic
- Compared every strategy's actual output on the same input, not just in theory
- Built a **FAISS vector store per strategy** and ran **5 real questions** against all six
- Found a genuine failure case (Q2) and used it to make a concrete case for real embeddings in production
- Tied every strategy back to a real "when would I actually use this" decision, specifically for healthcare/compliance documents

**Natural next steps**, if you want to keep going from here (not part of this
notebook, but worth knowing):
- Swap the TF-IDF vectors in Section 5 and Section 7 for real embeddings (Amazon
  Titan Embeddings, which you're already using in your multi-tenant platform,
  would be a direct, low-effort swap) and re-run Q2 to see whether real
  embeddings actually resolve the exclusion/withdrawal confusion.
- Build a small labeled evaluation set (10-20 question/correct-chunk pairs) and
  compute Recall@k properly, instead of eyeballing results.
- This is exactly the ground this notebook covers for **Module: Document
  ingestion and chunking** on your Clinical Intelligence Platform roadmap --
  the code here is a reasonable starting point to adapt directly into that module.
